In [3]:
import ssl
import nltk

# Solución temporal para que NLTK pueda descargar sus recursos
ssl._create_default_https_context = ssl._create_unverified_context

nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/josetanchezz/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/josetanchezz/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [4]:
import pandas as pd

RUTA_DATASET = "gutenberg_novels_dataset.csv"

# Cargar dataset
df = pd.read_csv(RUTA_DATASET)

# Verificar libros incluidos
display(df[["gutenberg_id", "title", "author", "text_length"]])

# Metadatos históricos para el reporte
informacion_libros = pd.DataFrame({
    "Título": [
        "Pride and Prejudice",
        "Frankenstein; or, The Modern Prometheus",
        "Dracula"
    ],
    "Autor(a)": [
        "Jane Austen",
        "Mary Shelley",
        "Bram Stoker"
    ],
    "Año / época aproximada de publicación": [
        "1813 — Inglaterra, época georgiana / Regencia",
        "1818 — Romanticismo inglés (edición revisada: 1831)",
        "1897 — Final de la época victoriana"
    ]
})

display(informacion_libros)

,gutenberg_id,title,author,text_length
0,1342,Pride and Prejudice,Jane Austen,728392
1,84,Frankenstein,Mary Shelley,419290
2,345,Dracula,Bram Stoker,845805


,Título,Autor(a),Año / época aproximada de publicación
0,Pride and Prejudice,Jane Austen,"1813 — Inglaterra, época georgiana / Regencia"
1,"Frankenstein; or, The Modern Prometheus",Mary Shelley,1818 — Romanticismo inglés (edición revisada: ...
2,Dracula,Bram Stoker,1897 — Final de la época victoriana


In [5]:
import random
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize

# Descargar recurso de NLTK para segmentar oraciones
nltk.download("punkt")
nltk.download("punkt_tab")  # Necesario en versiones recientes de NLTK

SEMILLA = 42
TAMANO_MUESTRA = 100

def limpiar_texto_gutenberg(texto):
    """
    Elimina, cuando están presentes, las cabeceras y notas finales
    añadidas por Project Gutenberg. Así los conteos representan
    principalmente el contenido de la novela.
    """
    inicio = texto.find("*** START OF")
    if inicio != -1:
        salto_linea = texto.find("\n", inicio)
        texto = texto[salto_linea:].strip()

    fin = texto.find("*** END OF")
    if fin != -1:
        texto = texto[:fin].strip()

    return texto

resultados = []
muestras = {}

for _, fila in df.iterrows():
    titulo = fila["title"]
    texto_limpio = limpiar_texto_gutenberg(fila["text"])

    # Segmentación en oraciones
    oraciones = sent_tokenize(texto_limpio)

    # Tokenización de todo el libro
    tokens_libro = word_tokenize(texto_limpio)

    # Muestra aleatoria reproducible de 100 oraciones
    generador = random.Random(SEMILLA)
    muestra_oraciones = generador.sample(
        oraciones,
        k=min(TAMANO_MUESTRA, len(oraciones))
    )

    # Tokenización de la muestra
    tokens_muestra = [
        token
        for oracion in muestra_oraciones
        for token in word_tokenize(oracion)
    ]

    # Guardar la muestra: se utilizará en las secciones 2 y 3
    muestras[titulo] = {
        "oraciones": muestra_oraciones,
        "tokens": tokens_muestra
    }

    resultados.append({
        "Libro": titulo,
        "Total de oraciones": len(oraciones),
        "Total de tokens": len(tokens_libro),
        "Oraciones en la muestra": len(muestra_oraciones),
        "Tokens en la muestra": len(tokens_muestra)
    })

tabla_comparativa = pd.DataFrame(resultados)
display(tabla_comparativa)

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/josetanchezz/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/josetanchezz/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


,Libro,Total de oraciones,Total de tokens,Oraciones en la muestra,Tokens en la muestra
0,Pride and Prejudice,6140,152022,100,2552
1,Frankenstein,3315,85708,100,2347
2,Dracula,8497,191692,100,2083
